## Главная задача

Из накопительных показаний получить **суточное потребление** (расход за день =
разница показаний между соседними днями) и найти в потреблении аномалии.

## Суточные потребления

In [1]:
import pandas as pd

df_pokazaniya = pd.read_csv(
    "pokazaniya.csv",
    sep=";",
    encoding="utf-8-sig",
    dtype={"anon_id": str},
    parse_dates=["data"],
)

df_pokazaniya = df_pokazaniya.sort_values(["anon_id", "data"])

df_pokazaniya["dni_mezhdu"] = df_pokazaniya.groupby("anon_id")["data"].diff().dt.days
df_pokazaniya["raznica"] = df_pokazaniya.groupby("anon_id")["pokazanie"].diff().round(3)

# Расходом именно за одни сутки считаем разницу только соседних дат.
df_pokazaniya["rashod_za_sutki"] = df_pokazaniya["raznica"].where(df_pokazaniya["dni_mezhdu"] == 1)

In [2]:
df_pokazaniya.to_csv('pokazanaiya_res.csv', sep=";", index=False, encoding="utf-8-sig")

## Что найти (аномалии потребления)

Для каждого пункта: ваш критерий (порог) + сколько приборов + список/примеры
самых ярких `anon_id`.

## 1. **Отрицательный расход** — показание уменьшилось (счётчик «пошёл назад»).
   Что это может означать?
   накопительное показание обычно должно увеличиваться или оставаться прежним. Его уменьшение может быть связано со сбросом счётчика, заменой прибора, ошибкой передачи данных, неисправностью или обратным движением счётчика. По отрицательной разнице нельзя определить точную причину — нужно проверить события «сброс» и историю прибора.

In [3]:
# Строки, где показание уменьшилось
df_minus_rasxod = df_pokazaniya[
    df_pokazaniya["raznica"] < 0
]

# Количество уникальных приборов
print("Приборов:", df_minus_rasxod["anon_id"].nunique())

# Количество случаев уменьшения
print("Случаев:", len(df_minus_rasxod))

# Количество падений показаний у каждого прибора
top_minus = (
    df_minus_rasxod.groupby("anon_id")
    .size()
    .sort_values(ascending=False)
    .head(5)
    .rename("kolichestvo_padeniy")
)

print('В первую очередь стоит проверить приборы приведенные ниже у них самое большое количество падений')
display(top_minus.to_frame())



Приборов: 102
Случаев: 269
В первую очередь стоит проверить приборы приведенные ниже у них самое большое количество падений


,kolichestvo_padeniy
anon_id,
2030325,13
7931977,12
3917499,10
8190741,10
7293232,8


## 2. **Всплески** — аномально большой расход за сутки. Дайте и абсолютный порог,
   и относительный (например, в N раз больше обычного для этого прибора).

Критерии:
- Абсолютный: расход больше 1 м³ за сутки — 1000 литров.
- Относительный: расход больше обычного для этого прибора в 5 раз, при этом больше 0.1 м³ — чтобы не считать всплеском совсем маленькие изменения.
- Обычный расход: медиана положительных суточных расходов прибора. Например, для значений 0.1, 0.2, 0.3 медиана равна 0.2 м³.

In [4]:
# Обычный порого медиана
df_pokazaniya["obychnyy_rashod"] = (
    df_pokazaniya["rashod_za_sutki"]
    .where(df_pokazaniya["rashod_za_sutki"] > 0)
    .groupby(df_pokazaniya["anon_id"])
    .transform("median")
)

In [5]:
# Абсолютный порог больше 1м^3
mask_abs = df_pokazaniya["rashod_za_sutki"] > 1
df_vspleski_abs = df_pokazaniya[mask_abs]

print("Приборов:", df_vspleski_abs["anon_id"].nunique())
print("Случаев:", len(df_vspleski_abs))

display(
    df_vspleski_abs.groupby("anon_id")
    .size()
    .sort_values(ascending=False)
    .head(5)
    .rename("kolichestvo_1m3")
    .to_frame()
)

Приборов: 367
Случаев: 7697


,kolichestvo_1m3
anon_id,
1606847,282
7199412,279
5754693,254
1648954,241
3161059,239


In [6]:
# Относительный порог превышающий в 5 раз (можно считать как всплески)
mask_rel = (
    (df_pokazaniya["rashod_za_sutki"]
     > 5 * df_pokazaniya["obychnyy_rashod"])
    & (df_pokazaniya["rashod_za_sutki"] > 0.1)
)

df_vspleski_rel = df_pokazaniya[mask_rel]

print("Приборов:", df_vspleski_rel["anon_id"].nunique())
print("Случаев:", len(df_vspleski_rel))

display(
    df_vspleski_rel.groupby("anon_id")
    .size()
    .sort_values(ascending=False)
    .head(5)
    .rename("kolichestvo_vspleskov")
    .to_frame()
)

Приборов: 750
Случаев: 4120


,kolichestvo_vspleskov
anon_id,
2158929,137
3190895,105
6675553,93
3036541,76
7626821,52


Всплеском считаем то что одновременно превышает относительный порог и абсолютный порого(для некоторых 1м3 может быть практически нормой)

In [7]:
df_vspleski_both = df_pokazaniya[mask_abs & mask_rel]

print("Приборов:", df_vspleski_both["anon_id"].nunique())
print("Случаев:", len(df_vspleski_both))

display(
    df_vspleski_both.groupby("anon_id")
    .size()
    .sort_values(ascending=False)
    .head(5)
    .rename("kolichestvo_vspleskov")
    .to_frame()
)

Приборов: 210
Случаев: 917


,kolichestvo_vspleskov
anon_id,
3036541,57
2030325,48
3743114,35
1278937,29
8727125,24


## 3 **Нереальные значения** — физически невозможный расход (десятки и тысячи м³ в сутки). Найдите такие приборы.

In [8]:
df_nereal = df_pokazaniya[
    df_pokazaniya["rashod_za_sutki"] > 10
]

print("Приборов:", df_nereal["anon_id"].nunique())
print("Случаев:", len(df_nereal))

# Самые большие расходы
print(
    df_nereal.nlargest(10, "rashod_za_sutki")[
        ["anon_id", "data", "rashod_za_sutki"]
    ].to_string(index=False)
)

Приборов: 63
Случаев: 379
anon_id       data  rashod_za_sutki
2491162 2026-07-09      3005503.868
8908023 2026-09-07      2684039.253
2159987 2025-10-17         2834.770
7891133 2025-10-17         2124.960
8973572 2025-10-17          608.136
5359103 2026-07-05          530.317
2030325 2026-07-20          425.916
4964924 2026-09-20          401.047
7247006 2026-09-29          364.053
8498147 2026-09-29          363.971


## 4 **Утечка** — непрерывное потребление без «нулевых» суток длительное время
   (прибор никогда не «отдыхает»). Ваш критерий и сколько.

In [9]:
# Положительный расход за сутки
mask = df_pokazaniya["rashod_za_sutki"] > 0

# Новая серия при нарушении условия или смене прибора
seriya = (
    ~mask
    | df_pokazaniya["anon_id"].ne(df_pokazaniya["anon_id"].shift())
).cumsum()
# Длина каждой серии положительного расхода
serii = (
    df_pokazaniya[mask]
    .groupby([df_pokazaniya["anon_id"], seriya])
    .size()
)

# Серии длительностью не менее 180 суток
utechki = serii[serii >= 180]
# Самая длинная подходящая серия каждого прибора
pribory_utechki = (
    utechki.groupby("anon_id")
    .max()
    .sort_values(ascending=False)
)
print("Приборов:", len(pribory_utechki))

print(
    pribory_utechki.head(5)
    .rename("sutok_podryad")
    .reset_index()
    .to_string(index=False)
)

Приборов: 79
anon_id  sutok_podryad
1968874            365
7496751            364
2931056            364
6554606            353
1282534            332


## 5 **Нулевое потребление** длительно, хотя прибор продолжает слать пакеты (пустая квартира / застрявший счётчик). Порог и количество.

In [10]:
# Нулевой суточный расход, но пакеты приходят
mask_zero = (
    (df_pokazaniya["rashod_za_sutki"] == 0)
    & (df_pokazaniya["paketov_za_sutki"] > 0)
)

# Номера серий
seriya_zero = (
    ~mask_zero
    | df_pokazaniya["anon_id"].ne(df_pokazaniya["anon_id"].shift())
).cumsum()

# Длина каждой серии
serii_zero = (
    df_pokazaniya[mask_zero]
    .groupby([df_pokazaniya["anon_id"], seriya_zero])
    .size()
)

# Серии от 30 суток
dlitelnyy_zero = serii_zero[serii_zero >= 30]

# Самая длинная серия каждого прибора
pribory_zero = (
    dlitelnyy_zero.groupby("anon_id")
    .max()
    .sort_values(ascending=False)
)

print("Приборов:", len(pribory_zero))

print(
    pribory_zero.head(5)
    .rename("sutok_bez_rashoda")
    .reset_index()
    .to_string(index=False)
)

Приборов: 694
anon_id  sutok_bez_rashoda
9686922                364
7558835                364
9354769                360
5374718                331
1547698                322


## 6 **Ступенька** — резкое устойчивое изменение уровня потребления (например, средний расход во второй половине периода в разы выше первой).

Критерии:
- Период каждого прибора делим пополам по его собственным датам (от первой до последней записи).
- Берём только корректные суточные расходы: от 0 до 10 м³ — отрицательные и нереальные значения (пункты 1 и 3) исказили бы среднее.
- В каждой половине не меньше 30 суток с расходом, иначе сравнивать нечего.
- Ступенька вверх: средний расход второй половины **в 3 раза и больше** выше первой. Ступенька вниз — наоборот.
- Высокий уровень не меньше 0.05 м³/сут (50 литров), чтобы не считать ступенькой переход 0.001 → 0.004.
- **Устойчивость:** в «высокой» половине не меньше 50% суток выше середины между уровнями. Так отсекаем случаи, где среднее подняли несколько всплесков, а не новый уровень потребления.

In [11]:
# Только корректные суточные расходы
df_step = df_pokazaniya[df_pokazaniya["rashod_za_sutki"].between(0, 10)].copy()

# Середина периода каждого прибора
data_min = df_step.groupby("anon_id")["data"].transform("min")
data_max = df_step.groupby("anon_id")["data"].transform("max")
df_step["polovina"] = (df_step["data"] > data_min + (data_max - data_min) / 2).map({False: 1, True: 2})

# Средний расход и количество суток в каждой половине
polovini = (
    df_step.groupby(["anon_id", "polovina"])["rashod_za_sutki"]
    .agg(["mean", "count"])
    .unstack()
)
polovini.columns = [f"{a}_{b}" for a, b in polovini.columns]
polovini = polovini[(polovini["count_1"] >= 30) & (polovini["count_2"] >= 30)]

print("Приборов с достаточными данными:", len(polovini))

Приборов с достаточными данными: 2328


In [12]:
# Устойчивость: доля суток выше середины между уровнями
seredina = ((polovini["mean_1"] + polovini["mean_2"]) / 2).rename("seredina")
df_step = df_step.join(seredina, on="anon_id", how="inner")
df_step["vyshe"] = df_step["rashod_za_sutki"] > df_step["seredina"]

dolya = df_step.groupby(["anon_id", "polovina"])["vyshe"].mean().unstack()
polovini["vyshe_1"] = dolya[1]
polovini["vyshe_2"] = dolya[2]

polovini["izmenenie"] = polovini["mean_2"] - polovini["mean_1"]

mask_up = (
    (polovini["mean_2"] >= 3 * polovini["mean_1"])
    & (polovini["mean_2"] >= 0.05)
    & (polovini["vyshe_2"] >= 0.5)
)
mask_down = (
    (polovini["mean_1"] >= 3 * polovini["mean_2"])
    & (polovini["mean_1"] >= 0.05)
    & (polovini["vyshe_1"] >= 0.5)
)

stupenki_up = polovini[mask_up].sort_values("izmenenie", ascending=False)
stupenki_down = polovini[mask_down].sort_values("izmenenie")

kolonki = ["mean_1", "mean_2", "izmenenie", "vyshe_1", "vyshe_2"]

print("Ступенька вверх, приборов:", len(stupenki_up))
print(
    stupenki_up[kolonki].head(5)
    .round(3)
    .reset_index()
    .to_string(index=False)
)

print("Ступенька вниз, приборов:", len(stupenki_down))
print(
    stupenki_down[kolonki].head(5)
    .round(3)
    .reset_index()
    .to_string(index=False)
)

print("Всего приборов со ступенькой:", len(stupenki_up) + len(stupenki_down))

Ступенька вверх, приборов: 72
anon_id  mean_1  mean_2  izmenenie  vyshe_1  vyshe_2
3761383   0.038   2.136      2.098    0.000    0.926
1581018   0.621   2.505      1.885    0.158    0.744
2704749   0.047   0.492      0.445    0.043    0.794
4065919   0.008   0.433      0.425    0.000    0.700
9481978   0.159   0.563      0.405    0.138    0.776
Ступенька вниз, приборов: 19
anon_id  mean_1  mean_2  izmenenie  vyshe_1  vyshe_2
9996860   0.400   0.083     -0.317    0.727    0.125
8267143   0.257   0.015     -0.242    0.835    0.000
6170256   0.184   0.000     -0.184    0.763    0.000
6951752   0.234   0.070     -0.164    0.750    0.188
6675553   0.167   0.013     -0.154    0.589    0.016
Всего приборов со ступенькой: 91


Ступенька вниз (особенно до нуля) интереснее для проверки: это может быть выезд жильцов, но также застрявший счётчик или вмешательство — проверяем в пункте 7 вместе с событиями «магнит».
Ступенька вверх — новые жильцы / новый прибор / начало утечки (сверить с пунктом 4).

## 7 Сопоставьте падение/аномалии потребления с событиями «магнит» из `sobytiya.csv` — есть ли связь?

Событие «магнит» — прибор зафиксировал магнитное поле рядом с собой. Магнитом можно затормозить счётчик, чтобы он показывал меньше. Если так делают, после магнита расход должен падать.

Проверяем по шагам:
1. Чаще ли аномалии из пунктов 1–6 бывают у приборов с магнитом, чем у приборов без него.
2. Падает ли расход после первого магнита: средний суточный расход за 30 суток до и за 30 суток после.
3. Что происходит в сами дни с магнитом: приходит ли показание и есть ли расход.

В `sobytiya.csv` одно и то же событие за день записано по несколько раз — дубли убираем.

In [13]:
df_sobytiya_raw = pd.read_csv(
    "sobytiya.csv",
    sep=";",
    encoding="utf-8-sig",
    dtype={"anon_id": str},
    parse_dates=["data"],
)
df_sobytiya = df_sobytiya_raw.drop_duplicates()

df_magnit = df_sobytiya[df_sobytiya["sobytie"] == "магнит"]
vse_pribory = set(df_pokazaniya["anon_id"])
pribory_magnit = set(df_magnit["anon_id"])
pribory_bez_magnita = vse_pribory - pribory_magnit

print(f"Записей «магнит» в файле: {(df_sobytiya_raw['sobytie'] == 'магнит').sum()}")
print(f"После удаления дублей (один прибор — один день): {len(df_magnit)}")
print(f"Приборов с магнитом: {len(pribory_magnit)} из {len(vse_pribory)} ({len(pribory_magnit) / len(vse_pribory):.1%})")

Записей «магнит» в файле: 33365
После удаления дублей (один прибор — один день): 10164
Приборов с магнитом: 671 из 3000 (22.4%)


### Шаг 1. Аномалии у приборов с магнитом и без
Для каждой аномалии считаем, какая доля приборов в каждой группе её имеет. Если магнит связан с аномалией, в группе «с магнитом» доля должна быть заметно больше.

In [14]:
anomalii = {
    "Отрицательный расход (п.1)": set(df_minus_rasxod["anon_id"]),
    "Всплески (п.2)": set(df_vspleski_both["anon_id"]),
    "Нереальные значения (п.3)": set(df_nereal["anon_id"]),
    "Утечка (п.4)": set(pribory_utechki.index),
    "Длительный ноль (п.5)": set(pribory_zero.index),
    "Ступенька вверх (п.6)": set(stupenki_up.index),
    "Ступенька вниз (п.6)": set(stupenki_down.index),
}

sravnenie = pd.DataFrame(
    [
        {
            "Аномалия": nazvanie,
            "С магнитом, %": len(pribory & pribory_magnit) / len(pribory_magnit) * 100,
            "Без магнита, %": len(pribory & pribory_bez_magnita) / len(pribory_bez_magnita) * 100,
        }
        for nazvanie, pribory in anomalii.items()
    ]
)

print(sravnenie.round(1).to_string(index=False))

chashe = sravnenie[sravnenie["С магнитом, %"] > 1.5 * sravnenie["Без магнита, %"]]
if chashe.empty:
    print("\nНи одна аномалия не встречается у приборов с магнитом заметно чаще (в 1.5 раза и больше).")
else:
    print("\nЗаметно чаще у приборов с магнитом:", ", ".join(chashe["Аномалия"]))

                  Аномалия  С магнитом, %  Без магнита, %
Отрицательный расход (п.1)            2.5             3.6
            Всплески (п.2)            5.4             7.5
 Нереальные значения (п.3)            0.6             2.5
              Утечка (п.4)            2.8             2.6
     Длительный ноль (п.5)           16.7            25.0
     Ступенька вверх (п.6)            1.9             2.5
      Ступенька вниз (п.6)            0.6             0.6

Ни одна аномалия не встречается у приборов с магнитом заметно чаще (в 1.5 раза и больше).


### Шаг 2. Расход до и после первого магнита
Берём только приборы, у которых и до, и после магнита есть не меньше 10 суток с расходом. Падением считаем снижение среднего расхода в 2 раза и больше.

In [15]:
pervyy_magnit = df_magnit.groupby("anon_id")["data"].min().rename("pervyy_magnit")

df_m = df_pokazaniya[df_pokazaniya["rashod_za_sutki"].between(0, 10)].join(
    pervyy_magnit, on="anon_id", how="inner"
)
df_m["dney_ot_magnita"] = (df_m["data"] - df_m["pervyy_magnit"]).dt.days

do = df_m[df_m["dney_ot_magnita"].between(-30, -1)].groupby("anon_id")["rashod_za_sutki"].agg(["mean", "count"])
posle = df_m[df_m["dney_ot_magnita"].between(0, 29)].groupby("anon_id")["rashod_za_sutki"].agg(["mean", "count"])

do_posle = do.join(posle, lsuffix="_do", rsuffix="_posle", how="inner")
do_posle = do_posle[(do_posle["count_do"] >= 10) & (do_posle["count_posle"] >= 10)]

upal = do_posle[(do_posle["mean_do"] >= 2 * do_posle["mean_posle"]) & (do_posle["mean_do"] >= 0.05)]
vyros = do_posle[(do_posle["mean_posle"] >= 2 * do_posle["mean_do"]) & (do_posle["mean_posle"] >= 0.05)]

print(f"Удалось сравнить до/после: {len(do_posle)} приборов из {len(pribory_magnit)}")
print(f"Типичный (медианный) расход до магнита: {do_posle['mean_do'].median():.3f} м³/сут")
print(f"Типичный (медианный) расход после магнита: {do_posle['mean_posle'].median():.3f} м³/сут")
print(f"Расход упал в 2 раза и больше: {len(upal)} приборов")
print(f"Расход вырос в 2 раза и больше: {len(vyros)} приборов")

print("\nПриборы, где расход после магнита упал сильнее всего:")
print(
    upal.assign(padenie=upal["mean_do"] - upal["mean_posle"])
    .sort_values("padenie", ascending=False)
    .head(5)
    .reset_index()
    .rename(columns={"anon_id": "Прибор", "mean_do": "Расход до, м³/сут", "mean_posle": "Расход после, м³/сут"})
    [["Прибор", "Расход до, м³/сут", "Расход после, м³/сут"]]
    .round(3)
    .to_string(index=False)
)

po_mesyacam = pervyy_magnit.dt.to_period("M").value_counts().sort_index()
dolya_konec = po_mesyacam[po_mesyacam.index >= pd.Period("2026-08", "M")].sum() / len(pervyy_magnit)

print("\nКогда у приборов появился первый магнит:")
print(
    po_mesyacam.rename_axis("Месяц")
    .rename("Приборов")
    .reset_index()
    .to_string(index=False)
)
print(f"\nВ августе–сентябре 2026 (последние 2 месяца данных): {dolya_konec:.0%} приборов.")
print("Поэтому у большинства «после магнита» ещё слишком мало данных, и сравнить удалось не всех.")

Удалось сравнить до/после: 225 приборов из 671
Типичный (медианный) расход до магнита: 0.042 м³/сут
Типичный (медианный) расход после магнита: 0.042 м³/сут
Расход упал в 2 раза и больше: 7 приборов
Расход вырос в 2 раза и больше: 9 приборов

Приборы, где расход после магнита упал сильнее всего:
 Прибор  Расход до, м³/сут  Расход после, м³/сут
5368489              0.190                 0.085
7032675              0.091                 0.000
5490175              0.090                 0.000
8122113              0.090                 0.002
5945886              0.082                 0.005

Когда у приборов появился первый магнит:
  Месяц  Приборов
2025-10         6
2025-11         2
2025-12         4
2026-01         4
2026-02        16
2026-03        31
2026-04         9
2026-05        27
2026-06        27
2026-07        43
2026-08       126
2026-09       376

В августе–сентябре 2026 (последние 2 месяца данных): 75% приборов.
Поэтому у большинства «после магнита» ещё слишком мало данных, и с

### Шаг 3. Что происходит в сами дни с магнитом
Смотрим только приборы с магнитом и только дни внутри их периода наблюдения (от первой до последней записи). Сравниваем дни с магнитом и обычные дни тех же приборов.

In [16]:
df_mp = df_pokazaniya[df_pokazaniya["anon_id"].isin(pribory_magnit)]
period = df_mp.groupby("anon_id")["data"].agg(["min", "max"])

# Полный календарь каждого прибора: от первой до последней записи
kalendar = pd.concat(
    pd.DataFrame({"anon_id": anon_id, "data": pd.date_range(r["min"], r["max"])})
    for anon_id, r in period.iterrows()
)
kalendar = kalendar.merge(
    df_mp[["anon_id", "data", "rashod_za_sutki"]].assign(est_pokazanie=True),
    on=["anon_id", "data"], how="left",
).merge(
    df_magnit[["anon_id", "data"]].assign(magnit=True),
    on=["anon_id", "data"], how="left",
)
kalendar["est_pokazanie"] = kalendar["est_pokazanie"].fillna(False).astype(bool)
kalendar["magnit"] = kalendar["magnit"].fillna(False).astype(bool)

dni = kalendar.groupby("magnit").agg(
    dney=("data", "size"),
    net_pokazaniya=("est_pokazanie", lambda x: (~x).mean() * 100),
    nulevoy=("rashod_za_sutki", lambda x: (x.dropna() == 0).mean() * 100),
    sredniy=("rashod_za_sutki", lambda x: x.dropna().clip(0, 10).mean()),
)
dni.index = dni.index.map({True: "День с магнитом", False: "Обычный день"})

print(
    dni.rename_axis("Тип дня")
    .rename(columns={
        "dney": "Дней",
        "net_pokazaniya": "Нет показания, %",
        "nulevoy": "Нулевой расход, %",
        "sredniy": "Средний расход, м³/сут",
    })
    .round(3)
    .reset_index()
    .to_string(index=False)
)

        Тип дня   Дней  Нет показания, %  Нулевой расход, %  Средний расход, м³/сут
   Обычный день 171790            24.623             28.389                   0.154
День с магнитом   9818            41.729             34.960                   0.153


### Итоговый вывод по пункту 7

In [17]:
s_m = sravnenie.set_index("Аномалия")
net_m = dni.loc["День с магнитом", "net_pokazaniya"]
net_o = dni.loc["Обычный день", "net_pokazaniya"]
nol_m = dni.loc["День с магнитом", "nulevoy"]
nol_o = dni.loc["Обычный день", "nulevoy"]
do_nulya = upal[upal["mean_posle"] < 0.01].index

print(f"""Вывод: магнит НЕ связан с падением расхода, но связан с пропусками показаний.

1. Магнит фиксировался у {len(pribory_magnit)} приборов из {len(vse_pribory)} ({len(pribory_magnit) / len(vse_pribory):.0%}).

2. Аномалий у этих приборов не больше, чем у остальных, а по большинству пунктов даже меньше.
   Например, длительный ноль: {s_m.loc['Длительный ноль (п.5)', 'С магнитом, %']:.1f}% у приборов с магнитом против {s_m.loc['Длительный ноль (п.5)', 'Без магнита, %']:.1f}% без него;
   ступенька вниз: {s_m.loc['Ступенька вниз (п.6)', 'С магнитом, %']:.1f}% против {s_m.loc['Ступенька вниз (п.6)', 'Без магнита, %']:.1f}%.

3. Расход после первого магнита не падает. Из {len(do_posle)} приборов, которые удалось сравнить,
   типичный расход до — {do_posle['mean_do'].median():.3f} м³/сут, после — {do_posle['mean_posle'].median():.3f} м³/сут.
   Упал в 2+ раза у {len(upal)}, вырос в 2+ раза у {len(vyros)} — это обычный разброс, а не эффект магнита.

4. Зато в дни с магнитом показание не приходит в {net_m:.0f}% случаев против {net_o:.0f}% в обычные дни,
   то есть пропуски примерно в {net_m / net_o:.1f} раза чаще. Нулевых суток тоже немного больше: {nol_m:.0f}% против {nol_o:.0f}%.
   Возможное объяснение: магнит или помехи мешают прибору передавать данные.

5. Ограничение: {dolya_konec:.0%} приборов получили первый магнит в августе–сентябре 2026, в конце периода,
   поэтому данных «после магнита» мало. Вывод стоит перепроверить на более длинном периоде.

На ручную проверку — приборы, где расход после магнита упал почти до нуля: {', '.join(do_nulya)}.""")

Вывод: магнит НЕ связан с падением расхода, но связан с пропусками показаний.

1. Магнит фиксировался у 671 приборов из 3000 (22%).

2. Аномалий у этих приборов не больше, чем у остальных, а по большинству пунктов даже меньше.
   Например, длительный ноль: 16.7% у приборов с магнитом против 25.0% без него;
   ступенька вниз: 0.6% против 0.6%.

3. Расход после первого магнита не падает. Из 225 приборов, которые удалось сравнить,
   типичный расход до — 0.042 м³/сут, после — 0.042 м³/сут.
   Упал в 2+ раза у 7, вырос в 2+ раза у 9 — это обычный разброс, а не эффект магнита.

4. Зато в дни с магнитом показание не приходит в 42% случаев против 25% в обычные дни,
   то есть пропуски примерно в 1.7 раза чаще. Нулевых суток тоже немного больше: 35% против 28%.
   Возможное объяснение: магнит или помехи мешают прибору передавать данные.

5. Ограничение: 75% приборов получили первый магнит в августе–сентябре 2026, в конце периода,
   поэтому данных «после магнита» мало. Вывод стоит перепроверит

## 8 Собственные находки по потреблению

Для находок подключаем `pribory.csv` (тип, базовая станция, год выпуска) и для каждого прибора считаем средний расход и долю нулевых суток. Берём только корректные суточные расходы от 0 до 10 м³.

**«Мёртвый» прибор** — прибор, у которого 95% суток и больше с нулевым расходом, то есть он практически никогда ничего не считает.

In [18]:
df_pribory = pd.read_csv(
    "pribory.csv",
    sep=";",
    encoding="utf-8-sig",
    dtype={"anon_id": str},
)
df_pribory["god_vypuska"] = df_pribory["mesyac_vypuska"].str[:4]

df_ok = df_pokazaniya[df_pokazaniya["rashod_za_sutki"].between(0, 10)]

po_priboram = (
    df_ok.groupby("anon_id")
    .agg(
        sredniy=("rashod_za_sutki", "mean"),
        dolya_nol=("rashod_za_sutki", lambda x: (x == 0).mean()),
    )
    .join(df_pribory.set_index("anon_id"))
)
po_priboram["mertvyy"] = po_priboram["dolya_nol"] >= 0.95
po_priboram["nerealnyy"] = po_priboram.index.isin(df_nereal["anon_id"])

print(f"Приборов в расчёте: {len(po_priboram)}")
print(f"«Мёртвых» приборов: {po_priboram['mertvyy'].sum()} ({po_priboram['mertvyy'].mean():.0%})")

Приборов в расчёте: 2999
«Мёртвых» приборов: 732 (24%)


### Находка 1. Четверть приборов вообще не потребляет — почти все это новые приборы одного типа и одна базовая станция
Смотрим, у каких типов приборов и на каких базовых станциях «мёртвых» приборов больше всего.

In [19]:
def svodka(kolonka, nazvanie):
    return (
        po_priboram.groupby(kolonka)
        .agg(
            vsego=("mertvyy", "size"),
            mertvyh=("mertvyy", "sum"),
            dolya=("mertvyy", "mean"),
        )
        .assign(dolya=lambda x: x["dolya"] * 100)
        .rename_axis(nazvanie)
        .rename(columns={"vsego": "Приборов", "mertvyh": "«Мёртвых»", "dolya": "«Мёртвых», %"})
        .round(1)
        .reset_index()
        .to_string(index=False)
    )

print("По типу прибора:")
print(svodka("tip", "Тип"))
print("\nПо году выпуска:")
print(svodka("god_vypuska", "Год"))
print("\nПо базовой станции:")
print(svodka("bs", "Станция"))

mertvye = po_priboram[po_priboram["mertvyy"]]
pokazanie_mertvyh = df_pokazaniya[df_pokazaniya["anon_id"].isin(mertvye.index)].groupby("anon_id")["pokazanie"].max()
print(f"\nТипичное (медианное) показание «мёртвого» прибора: {pokazanie_mertvyh.median():.3f} м³ — счётчик почти на нуле, то есть новый.")
print(f"Из {len(pribory_zero)} приборов пункта 5 (длительный ноль) «мёртвые» весь период: {len(set(pribory_zero.index) & set(mertvye.index))}")

По типу прибора:
               Тип  Приборов  «Мёртвых»  «Мёртвых», %
             AQUA2       391         30           7.7
  aqua2_nvt_wa_stm      1594        645          40.5
         aqua2_stm      1011         57           5.6
aqua2set_fudan_stm         3          0           0.0

По году выпуска:
 Год  Приборов  «Мёртвых»  «Мёртвых», %
2018        69          9          13.0
2019       260         19           7.3
2020       193          6           3.1
2021       213         14           6.6
2022       508         25           4.9
2023       260         23           8.8
2024       392        103          26.3
2025       632        354          56.0
2026       472        179          37.9

По базовой станции:
Станция  Приборов  «Мёртвых»  «Мёртвых», %
  BS-01       300         52          17.3
  BS-02       300         40          13.3
  BS-03       300         55          18.3
  BS-04       300        287          95.7
  BS-05       300         90          30.0
  BS-06       30

### Находка 2. Нереальные значения почти целиком дают старые приборы AQUA2
Сравниваем долю приборов с нереальными значениями (пункт 3) по типам.

In [20]:
nerealnye_po_tipu = (
    po_priboram.groupby("tip")
    .agg(
        vsego=("nerealnyy", "size"),
        s_nerealnymi=("nerealnyy", "sum"),
        dolya=("nerealnyy", "mean"),
    )
    .assign(dolya=lambda x: x["dolya"] * 100)
)

print(
    nerealnye_po_tipu.rename_axis("Тип")
    .rename(columns={"vsego": "Приборов", "s_nerealnymi": "С нереальными значениями", "dolya": "Доля, %"})
    .round(1)
    .reset_index()
    .to_string(index=False)
)

aqua2 = po_priboram[po_priboram["tip"] == "AQUA2"]
print(f"\nAQUA2 — {len(aqua2) / len(po_priboram):.0%} всех приборов, выпуск {aqua2['god_vypuska'].min()}–{aqua2['god_vypuska'].max()} гг.,")
print(f"но на них приходится {aqua2['nerealnyy'].sum()} из {po_priboram['nerealnyy'].sum()} приборов с нереальными значениями.")

               Тип  Приборов  С нереальными значениями  Доля, %
             AQUA2       391                        56     14.3
  aqua2_nvt_wa_stm      1594                         0      0.0
         aqua2_stm      1011                         7      0.7
aqua2set_fudan_stm         3                         0      0.0

AQUA2 — 13% всех приборов, выпуск 2018–2021 гг.,
но на них приходится 56 из 63 приборов с нереальными значениями.


### Находка 3. «Сезонное» падение расхода — ложное, его создают новые «мёртвые» приборы
Если считать медианный расход по всем приборам, он падает от осени к лету. Но в течение года к сети подключали новые приборы, многие из которых «мёртвые». Поэтому сравниваем со стабильной группой: приборы, у которых есть данные во все 12 месяцев, без «мёртвых». Октябрь 2026 не берём — там только один день.

In [21]:
df_mes = df_ok[df_ok["data"] < "2026-10-01"].assign(mesyac=lambda x: x["data"].dt.to_period("M"))

mesyacev_s_dannymi = df_mes.groupby("anon_id")["mesyac"].nunique()
stabilnye = mesyacev_s_dannymi[mesyacev_s_dannymi == 12].index.difference(mertvye.index)

po_mesyacam = pd.DataFrame({
    "Приборов (все)": df_mes.groupby("mesyac")["anon_id"].nunique(),
    "Медиана, все приборы": df_mes.groupby("mesyac")["rashod_za_sutki"].median(),
    "Медиана, стабильная группа": df_mes[df_mes["anon_id"].isin(stabilnye)].groupby("mesyac")["rashod_za_sutki"].median(),
})

print(f"Стабильная группа: {len(stabilnye)} приборов\n")
print(po_mesyacam.rename_axis("Месяц").round(3).reset_index().to_string(index=False))

Стабильная группа: 1497 приборов

  Месяц  Приборов (все)  Медиана, все приборы  Медиана, стабильная группа
2025-10            2032                 0.047                       0.065
2025-11            2057                 0.049                       0.070
2025-12            2178                 0.047                       0.070
2026-01            2296                 0.040                       0.069
2026-02            2363                 0.036                       0.072
2026-03            2397                 0.034                       0.073
2026-04            2422                 0.034                       0.072
2026-05            2436                 0.028                       0.065
2026-06            2420                 0.031                       0.068
2026-07            2495                 0.030                       0.068
2026-08            2598                 0.026                       0.064
2026-09            2949                 0.030                       0.071


### Итоговый вывод по пункту 8

In [22]:
vse_m = po_mesyacam["Медиана, все приборы"]
stab_m = po_mesyacam["Медиана, стабильная группа"]
nvt = po_priboram[po_priboram["tip"] == "aqua2_nvt_wa_stm"]
drugie = po_priboram[po_priboram["tip"] != "aqua2_nvt_wa_stm"]
bs04 = po_priboram[po_priboram["bs"] == "BS-04"]

print(f"""1. «Мёртвые» приборы. {len(mertvye)} приборов из {len(po_priboram)} ({len(mertvye) / len(po_priboram):.0%}) не потребляют воду вообще (95%+ нулевых суток),
   а их показание — около {pokazanie_mertvyh.median():.2f} м³, то есть счётчик почти новый.
   Это в основном новые приборы типа aqua2_nvt_wa_stm (2023–2026 гг.): среди них «мёртвых» {nvt['mertvyy'].mean():.0%}, у остальных типов — {drugie['mertvyy'].mean():.0%}.
   На станции BS-04 «мёртвые» {bs04['mertvyy'].mean():.0%} приборов — похоже на целый объект, где приборы поставили, но водой ещё не пользуются
   (новостройка) или они не подключены. Это объясняет {len(set(pribory_zero.index) & set(mertvye.index))} из {len(pribory_zero)} приборов пункта 5:
   это не «застрявшие» счётчики, а неиспользуемые. Такие приборы лучше проверять по объекту целиком, а не по одному.

2. Нереальные значения — проблема старых приборов. Тип AQUA2 (выпуск {aqua2['god_vypuska'].min()}–{aqua2['god_vypuska'].max()}) — это {len(aqua2) / len(po_priboram):.0%} всех приборов,
   но на него приходится {aqua2['nerealnyy'].sum()} из {po_priboram['nerealnyy'].sum()} приборов с нереальными значениями ({aqua2['nerealnyy'].mean():.0%} всех AQUA2).
   Скорее всего, это сбои старой электроники, а не реальный расход. AQUA2 — первые кандидаты на замену.

3. Сезонности в потреблении нет. По всем приборам медиана расхода падает с {vse_m.iloc[0]:.3f} до {vse_m.iloc[-1]:.3f} м³/сут,
   но это потому, что за год число приборов выросло с {po_mesyacam['Приборов (все)'].iloc[0]} до {po_mesyacam['Приборов (все)'].iloc[-1]}, и многие новые — «мёртвые».
   В стабильной группе ({len(stabilnye)} приборов) расход ровный весь год: от {stab_m.min():.3f} до {stab_m.max():.3f} м³/сут.
   Вывод: при сравнении периодов нужно смотреть на одни и те же приборы, иначе получится ложный тренд.""")

1. «Мёртвые» приборы. 732 приборов из 2999 (24%) не потребляют воду вообще (95%+ нулевых суток),
   а их показание — около 0.05 м³, то есть счётчик почти новый.
   Это в основном новые приборы типа aqua2_nvt_wa_stm (2023–2026 гг.): среди них «мёртвых» 40%, у остальных типов — 6%.
   На станции BS-04 «мёртвые» 96% приборов — похоже на целый объект, где приборы поставили, но водой ещё не пользуются
   (новостройка) или они не подключены. Это объясняет 470 из 694 приборов пункта 5:
   это не «застрявшие» счётчики, а неиспользуемые. Такие приборы лучше проверять по объекту целиком, а не по одному.

2. Нереальные значения — проблема старых приборов. Тип AQUA2 (выпуск 2018–2021) — это 13% всех приборов,
   но на него приходится 56 из 63 приборов с нереальными значениями (14% всех AQUA2).
   Скорее всего, это сбои старой электроники, а не реальный расход. AQUA2 — первые кандидаты на замену.

3. Сезонности в потреблении нет. По всем приборам медиана расхода падает с 0.047 до 0.030 м³/сут,
   н